# Step 2: Data preparation

**Why an investor cares.** Diligence data arrives messy, and the buyer's adviser has to show that nothing built on it was distorted by that mess. A duplicated invoice, a sign error or a mislabelled one-off fee flows straight into growth rates, retention and the multiple paid. Profiling comes before cleaning because a fix can only be justified against a problem that has been measured: how many rows, how many pounds, in which years.

This notebook covers:

- **2.1 Load**: checks on the typed `raw_*` tables built by `sql/00_load.sql`.
- **2.2 Profile**: renders the `prof_*` tables built by `sql/01_profile.sql` into `docs/data_profile.md`.
- **2.3 Clean**: summarises `sql/02_clean.sql` and writes `docs/cleaning_log.md` from the `cleaning_actions` table.
- **2.4 Revenue cube**: summarises `sql/03_revenue_cube.sql`, the anomaly switch and the source-to-cube walk.
- **2.5 Documentation**: writes `docs/data_dictionary.md`, `docs/qa_log.md` and the generated blocks in `docs/metric_definitions.md` and `docs/alteryx_workflow.md`.

Cleaning fixes data errors only. Judgements about revenue quality belong in the analysis, so genuine revenue stays in the cube with flags.

Run `python run_all.py` from the repository root to rebuild the database before executing this notebook.

In [1]:
import pandas as pd
from nb_utils import connect, q, md_table, gbp, num, pct, DOCS, TABLES, replace_block

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

con = connect()
doc = []  # sections of docs/data_profile.md, in order

## 2.1 Load

`sql/00_load.sql` reads each CSV with an explicit schema (dates as `DATE`, money as `DECIMAL(14,2)`, rates as `DECIMAL(8,4)`, identifiers as `VARCHAR`). Type inference is switched off, so a value that does not fit its type stops the load instead of being silently coerced. Before fixing the types, every money column was checked as text and none has more than two decimal places, so nothing is rounded on load.

In [2]:
load = q(con, '''
    SELECT t.table_name, t.estimated_size AS row_count, count(c.column_name) AS columns
    FROM duckdb_tables() t JOIN duckdb_columns() c USING (table_name)
    WHERE t.table_name LIKE 'raw_%'
    GROUP BY ALL ORDER BY t.table_name
''')
load

,table_name,row_count,columns
0,raw_costs,192,6
1,raw_customers,2192,8
2,raw_invoices,92732,7
3,raw_management_accounts,48,9
4,raw_products,6,5
5,raw_subscriptions,3502,8


In [3]:
types = q(con, '''
    SELECT table_name, column_name, data_type FROM duckdb_columns()
    WHERE table_name LIKE 'raw_%' ORDER BY table_name, column_index
''')
types.groupby("data_type").size().rename("columns").to_frame()

,columns
data_type,
DATE,6
"DECIMAL(14,2)",14
"DECIMAL(8,4)",2
INTEGER,1
VARCHAR,20


## 2.2 Profile

Each block below reads one or more `prof_*` tables and adds a section to `docs/data_profile.md`. The descriptions are neutral: they state what the data shows and leave treatment to the cleaning step.

In [4]:
# 1. Tables, keys and nulls
pk = q(con, "SELECT * FROM prof_primary_keys")
cols = q(con, "SELECT * FROM prof_columns")
nulls = cols[cols.null_count > 0][["table_name", "column_name", "row_count", "null_count", "null_pct"]]

doc.append("## 1. Tables, primary keys and nulls\n")
doc.append(md_table(pk.assign(duplicate_keys=pk.row_count - pk.distinct_keys)))
doc.append("\nColumns with nulls (every other column is fully populated):\n")
doc.append(md_table(nulls, {"null_pct": pct}))
doc.append("\n`end_date` is blank for subscription lines that are still active, which is expected. "
           "`industry` is the only descriptive attribute with gaps.\n")
doc.append("<details><summary>Full column profile</summary>\n")
doc.append(md_table(cols.drop(columns=["ordinal"]), {"null_pct": pct}))
doc.append("\n</details>\n")
display(pk)
display(nulls)

,table_name,key_columns,row_count,distinct_keys,null_keys
0,raw_customers,customer_id,2192,2192,0
1,raw_products,product_id,6,6,0
2,raw_subscriptions,subscription_id,3502,3502,0
3,raw_invoices,invoice_id,92732,92732,0
4,raw_costs,"month, product_line",192,192,0
5,raw_management_accounts,month,48,48,0


,table_name,column_name,row_count,null_count,null_pct
9,raw_customers,industry,2192,132,6.02
39,raw_subscriptions,end_date,3502,2355,67.25


In [5]:
# 2. Date ranges
dr = q(con, "SELECT * FROM prof_date_ranges")
bs = q(con, "SELECT * FROM prof_invoice_before_signup").iloc[0]

doc.append("## 2. Date ranges\n")
doc.append("The data window is January 2022 to December 2025. `before_window` counts dates before 1 January 2022.\n")
doc.append(md_table(dr))
doc.append(f"""
- No invoice, cost or management-accounts date falls outside the window.
- Customer signups and subscription starts go back to {dr.loc[dr.column_name=='signup_date','min_date'].iloc[0]:%B %Y}: {int(dr.loc[dr.column_name=='signup_date','before_window'].iloc[0]):,} customers signed up before the window opens, so their early history is not in the invoice data.
- Subscription `end_date` values are always the 1st of a month. Invoice dates fall on any day of the month, and the day varies from month to month for the same customer and product.
- {int(bs.invoices):,} invoices ({gbp(bs.amount)}) for {int(bs.customers):,} customers are dated before the customer's `signup_date`, by {int(bs.min_days_before)} to {int(bs.max_days_before)} days (median {bs.median_days_before:.0f}). {'None of them falls' if bs.in_earlier_calendar_month == 0 else str(int(bs.in_earlier_calendar_month)) + ' of them fall'} in an earlier calendar month than the signup.
""")
display(dr)
display(bs.to_frame().T)

,table_name,column_name,min_date,max_date,null_dates,before_window,after_window,not_first_of_month
0,raw_costs,month,2022-01-01,2025-12-01,0,0,0,0
1,raw_customers,signup_date,2017-01-03,2025-12-27,0,1017,0,2124
2,raw_invoices,invoice_date,2022-01-01,2025-12-27,0,0,0,89356
3,raw_management_accounts,month,2022-01-01,2025-12-01,0,0,0,0
4,raw_subscriptions,end_date,2017-09-01,2025-11-01,2355,260,0,0
5,raw_subscriptions,start_date,2017-01-03,2025-12-27,0,1566,0,3074


,invoices,customers,amount,min_days_before,median_days_before,max_days_before,in_earlier_calendar_month
0,1424.0,806.0,2992625.42,1.0,8.0,26.0,0.0


In [6]:
# 3. Invoice amounts
am = q(con, "SELECT * FROM prof_amounts")
px = q(con, "SELECT * FROM prof_invoice_prefix")
od = q(con, "SELECT * FROM prof_overdue_by_year")
money = {c: gbp for c in ["total", "min_amount", "p01", "p25", "median", "mean", "p75", "p99", "max_amount", "negative_total"]}

doc.append("## 3. Invoice amounts\n")
doc.append("Distribution by status and revenue type (£):\n")
doc.append(md_table(am, money))
doc.append("\nInvoice-id prefix against status:\n")
doc.append(md_table(px, {"total": gbp}))
doc.append("\nOverdue billings by year (gross billings exclude credit notes):\n")
doc.append(md_table(od, {"gross_billings": gbp, "overdue_amount": gbp, "overdue_pct": pct}))
neg_paid = am[(am.status == "paid") & (am.revenue_type == "recurring")].iloc[0]
doc.append(f"""
- Every credit note carries the `CN` prefix and status `credited`, and every one is negative.
- {int(neg_paid.negative_count)} invoices with status `paid` carry a negative amount (total {gbp(neg_paid.negative_total, 2)}). No `overdue` invoice is negative. See section 7.
- No invoice has a zero amount.
""")
display(am)
display(od)

,status,revenue_type,invoices,total,min_amount,p01,p25,median,mean,p75,p99,max_amount,negative_count,negative_total,zero_count
0,credited,recurring,1084,-1.336469e+06,-52500.00,-3210.99,-1213.04,-756.00,-1232.91,-536.89,-455.99,-445.28,1084,-1336469.07,0
1,overdue,one-off,38,4.055554e+05,1245.89,1423.97,2872.13,3179.75,10672.51,4194.92,98314.86,104524.52,0,0.00,0
2,overdue,recurring,2750,4.058219e+06,445.28,459.63,534.50,715.90,1475.72,1193.63,3331.33,150736.25,0,0.00,0
3,paid,one-off,1182,8.182045e+06,938.96,1001.42,1907.59,3279.12,6922.20,4739.86,119375.17,147346.68,0,0.00,0
4,paid,recurring,87678,1.322141e+08,-2919.22,458.91,538.52,746.41,1507.95,1201.19,3410.04,150736.25,15,-13843.82,0
5,all,all,92732,1.435235e+08,-52500.00,-504.00,536.58,746.41,1547.72,1207.50,8811.18,150736.25,1099,-1350312.89,0


,year,gross_billings,overdue_invoices,overdue_amount,overdue_pct
0,2022,28564164.39,522,918707.94,3.22
1,2023,32537962.92,670,915646.20,2.81
2,2024,37583315.05,765,1092445.57,2.91
3,2025,46174479.60,831,1536974.82,3.33


In [7]:
# 4. Allowed values
dv = q(con, "SELECT * FROM prof_domain_values")
doc.append("## 4. Allowed values\n")
for (t, c), g in dv.groupby(["table_name", "column_name"], sort=False):
    doc.append(f"**{t}.{c}**: " + ", ".join(f"{v} ({n:,})" for v, n in zip(g.value, g.rows)) + "\n")
doc.append("All categorical values are spelled consistently: there are no case or spacing variants. "
           "`region` has ten UK regions and three EU countries marked `(EU)`. "
           "`industry` is null for the customers counted as `(null)`.\n")
dv

,table_name,column_name,value,rows
0,raw_costs,product_line,Analytics Add-on,48
1,raw_costs,product_line,Core Platform,48
2,raw_costs,product_line,Implementation Services,48
3,raw_costs,product_line,Payments Module,48
4,raw_customers,account_manager,F. Marsh,306
5,raw_customers,account_manager,E. Lindqvist,292
6,raw_customers,account_manager,G. Patel,285
7,raw_customers,account_manager,B. Novak,278
8,raw_customers,account_manager,C. Ferreira,263
9,raw_customers,account_manager,A. Okafor,258


In [8]:
# 5. Referential integrity and consistency
ri = q(con, "SELECT * FROM prof_referential_integrity")
rt = q(con, "SELECT * FROM prof_revenue_type_consistency")
doc.append("## 5. Referential integrity and revenue-type consistency\n")
doc.append(md_table(ri))
doc.append("\nInvoice `revenue_type` against the product's `revenue_type`:\n")
doc.append(md_table(rt, {"total": gbp}))
doc.append(f"""
- Every invoice and subscription points to a customer and product that exist.
- Every invoice's `revenue_type` agrees with its product.
- {int(ri.loc[ri.check_name.str.startswith('customers with no invoice'), 'orphan_rows'].iloc[0])} customers have no invoice in the window. None of them has a subscription line live at any point in the window, so their absence from the invoices is consistent with their contract history.
""")
display(ri)
display(rt)

,check_name,orphan_rows
0,invoices.customer_id not in customers,0
1,invoices.product_id not in products,0
2,subscriptions.customer_id not in customers,0
3,subscriptions.product_id not in products,0
4,costs.product_line not in products,0
5,customers with no subscription,0
6,customers with no invoice in the window,161
7,of which: a subscription line live at any point in the window,0


,invoice_revenue_type,product_revenue_type,product_line,invoices,total,consistent
0,one-off,one-off,Implementation Services,1220,8.587600e+06,True
1,recurring,recurring,Analytics Add-on,20144,1.541993e+07,True
2,recurring,recurring,Core Platform,58030,1.042095e+08,True
3,recurring,recurring,Payments Module,13338,1.530642e+07,True


In [9]:
# 6. Duplicate invoices
dup = q(con, "SELECT * FROM prof_duplicate_invoices")
dupy = q(con, "SELECT * FROM prof_duplicate_by_year")
mb = q(con, "SELECT * FROM prof_multi_billing_cells")
first_ids = dup.invoice_ids.map(lambda l: l[0])
second_ids = dup.invoice_ids.map(lambda l: l[-1])
max_inv = q(con, "SELECT max(invoice_id) AS m FROM raw_invoices WHERE invoice_id LIKE 'INV%'").m.iloc[0]

doc.append("## 6. Exact duplicate invoices\n")
doc.append("Duplicate key: every business field except `invoice_id` "
           "(customer_id, product_id, invoice_date, amount, revenue_type, status).\n")
doc.append(md_table(dupy, {"excess_amount": gbp, "largest_single_amount": gbp}))
doc.append(f"""
- {len(dup)} groups, each with exactly {int(dup.copies.max())} copies: {int((dup.copies - 1).sum())} excess rows worth {gbp(dup.excess_amount.sum(), 2)}.
- Every duplicated invoice has status `overdue`.
- 2023 is larger than the other years because one duplicated invoice is a Core Platform - Scale billing of {gbp(dupy.largest_single_amount.max(), 2)}.
- In every pair, the first `invoice_id` sits within the normal numbering ({first_ids.min()} to {first_ids.max()}) and the second falls in a separate block from {second_ids.min()} to {second_ids.max()}, which is the end of the `INV` sequence (highest `INV` id: {max_inv}).
- The only customer x product x month cells with more than one billing are these {int(mb.cells.sum())} pairs. There are no near-duplicates with differing amounts.
""")
doc.append("<details><summary>All duplicate groups</summary>\n")
doc.append(md_table(dup, {"amount": lambda v: gbp(v, 2), "excess_amount": lambda v: gbp(v, 2)}))
doc.append("\n</details>\n")
display(dupy)
dup.head(10)

,year,status,duplicate_groups,excess_rows,excess_amount,largest_single_amount
0,2022,overdue,9,9.0,8596.92,2614.29
1,2023,overdue,9,9.0,70158.24,64166.67
2,2024,overdue,8,8.0,7115.70,2823.24
3,2025,overdue,14,14.0,11086.40,1271.90


,customer_id,product_id,invoice_date,amount,revenue_type,status,copies,invoice_ids,excess_amount
0,C1988,2,2022-02-26,1150.00,recurring,overdue,2,"[INV754199, INV792712]",1150.00
1,C1867,1,2022-03-19,480.00,recurring,overdue,2,"[INV746768, INV792699]",480.00
2,C1234,1,2022-03-24,471.30,recurring,overdue,2,"[INV712318, INV792720]",471.30
3,C1239,2,2022-04-09,1127.64,recurring,overdue,2,"[INV712572, INV792710]",1127.64
4,C1397,4,2022-05-19,510.01,recurring,overdue,2,"[INV720848, INV792698]",510.01
5,C1283,3,2022-08-06,2614.29,recurring,overdue,2,"[INV715151, INV792697]",2614.29
6,C1232,5,2022-08-16,720.00,recurring,overdue,2,"[INV712256, INV792709]",720.00
7,C1272,2,2022-08-17,1051.85,recurring,overdue,2,"[INV714667, INV792711]",1051.85
8,C2165,1,2022-08-18,471.83,recurring,overdue,2,"[INV764484, INV792707]",471.83
9,C1328,3,2023-02-21,64166.67,recurring,overdue,2,"[INV717481, INV792719]",64166.67


In [10]:
# 7. Negative amounts and credit notes
neg = q(con, "SELECT * FROM prof_negative_non_credit")
cn = q(con, "SELECT * FROM prof_credit_notes")
money2 = lambda v: gbp(v, 2)
same_either_side = ((neg.prior_month_billing == -neg.amount) & (neg.next_month_billing == -neg.amount)).sum()

doc.append("## 7. Negative amounts and credit notes\n")
doc.append("Negative amounts on invoices that are not credit notes, with the same customer and product's "
           "billing in the same, prior and next month:\n")
doc.append(md_table(neg, {"amount": money2, "prior_month_billing": money2, "next_month_billing": money2}))
doc.append(f"""
- {len(neg)} rows, total {gbp(neg.amount.sum(), 2)}, all status `{"`, `".join(sorted(neg.status.unique()))}` and all on recurring products.
- In every case this is the only billing for that customer, product and month (`other_rows_same_month` = 0).
- In {same_either_side} of {len(neg)} cases the prior and next month's billing for the same line equals the same amount with a positive sign.
""")
doc.append("Credit notes by year, and whether each matches an original invoice on customer, product, date and absolute amount:\n")
doc.append(md_table(cn.assign(match_rate_pct=100 * cn.matched_to_original / cn.credit_notes),
                    {"credit_total": gbp, "match_rate_pct": pct}))
doc.append(f"\n- {int(cn.credit_notes.sum()):,} credit notes, {gbp(cn.credit_total.sum())} in total; "
           f"{int(cn.matched_to_original.sum()):,} ({100 * cn.matched_to_original.sum() / cn.credit_notes.sum():.1f}%) "
           "match an original invoice exactly, so each one fully reverses a single billing.\n")
display(neg)
display(cn)

,invoice_id,customer_id,product_id,invoice_date,amount,status,other_rows_same_month,prior_month_billing,next_month_billing
0,INV740991,C1773,5,2022-03-12,-702.47,paid,0,702.47,702.47
1,INV751844,C1947,2,2022-04-14,-1136.08,paid,0,1136.08,1136.08
2,INV717181,C1322,2,2022-07-11,-1030.98,paid,0,1030.98,1030.98
3,INV722989,C1437,3,2022-09-01,-2919.22,paid,0,2919.22,2919.22
4,INV754430,C1991,4,2023-01-13,-550.00,paid,0,550.00,550.00
5,INV710858,C1209,2,2023-07-03,-1098.00,paid,0,1098.00,1098.00
6,INV758329,C2058,1,2023-07-09,-470.96,paid,0,470.96,470.96
7,INV742515,C1795,5,2024-03-20,-626.35,paid,0,626.35,626.35
8,INV775990,C2444,2,2024-03-22,-1178.12,paid,0,1178.12,1178.12
9,INV757592,C2043,1,2024-04-05,-504.00,paid,0,504.00,504.00


,year,credit_notes,credit_total,matched_to_original,non_negative_credit_notes
0,2022,201,-329204.58,201,0
1,2023,258,-277053.69,258,0
2,2024,288,-321374.32,288,0
3,2025,337,-408836.48,337,0


In [11]:
# 8. Outliers
fen = q(con, "SELECT * FROM prof_outlier_fences")
rl = q(con, "SELECT * FROM prof_outlier_recurring_lines")
wc = q(con, "SELECT count(*) AS n FROM prof_outlier_within_customer").n.iloc[0]
it = q(con, "SELECT * FROM prof_implementation_timing")
off = q(con, "SELECT * FROM prof_implementation_off_cycle")

doc.append("## 8. Outliers\n")
doc.append("""Two rules, applied to positive invoices that are not credit notes:

- **Rule A (product level):** amount above Q3 + 3 x IQR of all invoices for the same product (Tukey's "far out" fence).
- **Rule B (customer level):** amount more than 3x, or less than a third of, the median of the same customer's invoices for the same recurring product.
""")
doc.append(md_table(fen, {"list_price": gbp, "q1": money2, "median_amount": money2, "q3": money2,
                          "upper_fence": money2, "amount_above": gbp}))
doc.append(f"""
**Recurring products.** Rule A flags {int(rl.invoices_above_fence.sum()):,} invoices ({gbp(rl.total.sum())}) on {len(rl)} customer x product lines belonging to {rl.customer_id.nunique()} customers. On every one of these lines, all invoices sit above the fence: the customer is billed at {rl.median_multiple_of_list.min():.0f}x to {rl.median_multiple_of_list.max():.0f}x list price in every month. Rule B flags {wc} invoices: no recurring line has a one-month spike or drop against its own history.
""")
doc.append(md_table(rl, {"list_price": gbp, "upper_fence": money2, "min_amount": money2, "median_amount": money2,
                         "max_amount": money2, "total": gbp, "median_multiple_of_list": lambda v: f"{v:.1f}x"}))
doc.append("\n**Implementation Services** (one-off). Timing of each invoice relative to the customer's signup month:\n")
doc.append(md_table(it, {"total": gbp, "min_amount": gbp, "median_amount": gbp, "max_amount": gbp}))
in_cycle = it[it.in_signup_month]
in_cycle_median = q(con, """
    SELECT median(i.amount) AS m FROM raw_invoices i JOIN raw_customers c USING (customer_id)
    WHERE i.product_id = '6' AND i.status <> 'credited'
      AND date_trunc('month', i.invoice_date) = date_trunc('month', c.signup_date)""").m.iloc[0]
doc.append(f"""
- {int(in_cycle.invoices.sum()):,} implementation invoices fall in the customer's signup month, one per new customer, median {gbp(in_cycle_median)}.
- {len(off)} implementation invoices ({gbp(off.amount.sum())}) fall outside the signup month. All are dated {", ".join(map(str, sorted(off.invoice_date.dt.year.unique())))}, range from {gbp(off.amount.min())} to {gbp(off.amount.max())} (median {gbp(off.amount.median())}), and are all above the Rule A fence.
- Their customers signed up between {off.signup_year.min()} and {off.signup_year.max()}; {int((off.signup_year < 2022).sum())} signed up before the data window.
- Their invoice ids form one block ({off.invoice_id.min()} to {off.invoice_id.max()}) that follows the last in-cycle implementation invoice.
- {int(off.recurring_billing_same_month.sum())} of the {len(off)} customers also have recurring billing in the same month; {int((off.other_implementation_invoices > 0).sum())} have another implementation invoice in the window.
""")
doc.append("<details><summary>Implementation invoices outside the signup month</summary>\n")
doc.append(md_table(off, {"amount": money2}))
doc.append("\n</details>\n")
display(fen)
display(it)

,product_id,product_name,list_price,invoices,q1,median_amount,q3,upper_fence,invoices_above,amount_above,customers_above
0,1,Core Platform - Starter,480.0,15843,476.71,489.56,518.14,642.43,0,0.00,0
1,2,Core Platform - Growth,1150.0,27313,1112.09,1150.00,1207.50,1493.73,0,0.00,0
2,3,Core Platform - Scale,2900.0,14162,2643.29,2769.63,2954.43,3887.85,289,26941506.51,6
3,4,Analytics Add-on,550.0,19890,520.64,546.23,576.50,744.08,218,4846635.12,5
4,5,Payments Module,720.0,13205,667.87,707.68,748.82,991.67,273,6277370.75,6
5,6,Implementation Services,0.0,1220,1914.72,3274.14,4737.00,13203.84,46,4089889.62,46


,year,in_signup_month,invoices,customers,total,min_amount,median_amount,max_amount,above_rule_a_fence,first_invoice_id,last_invoice_id
0,2022,True,242,242,983859.59,965.82,3276.38,11993.06,0,INV790388,INV790629
1,2023,True,310,310,1178203.80,938.96,3145.80,11871.46,0,INV790630,INV790939
2,2024,True,303,303,1144214.82,1009.07,3162.89,13315.31,1,INV790940,INV791242
3,2025,False,45,45,4076574.31,32292.09,88994.06,147346.68,45,INV791563,INV791607
4,2025,True,320,320,1204747.89,943.14,3179.29,11982.55,0,INV791243,INV791562


In [12]:
# 9. Subscriptions
sc = q(con, "SELECT * FROM prof_subscription_checks")
st = q(con, "SELECT * FROM prof_subscription_terms")
sp = q(con, "SELECT * FROM prof_subscription_prices")
doc.append("## 9. Subscriptions\n")
doc.append(md_table(sc, {"value": lambda v: num(v, 1) if not float(v).is_integer() else num(v)}))
doc.append("\nContract terms:\n")
doc.append(md_table(st, {"share_pct": pct}))
doc.append("\nMonthly price and discount by product:\n")
doc.append(md_table(sp, {"list_price": gbp, "min_price": money2, "median_price": money2, "max_price": money2,
                         "min_discount": lambda v: pct(100 * v), "median_discount": lambda v: pct(100 * v),
                         "max_discount": lambda v: pct(100 * v)}))
doc.append("""
- No line starts after it ends, and no customer has two active lines for the same product.
- Every month in which a recurring product is invoiced is covered by a subscription line for that customer and product.
- The maximum prices on Core Platform - Scale, Analytics Add-on and Payments Module are the large customer lines seen in section 8.
""")
display(sc); display(st); display(sp)

,metric,value
0,contract lines,3502.0
1,start_date after end_date,0.0
2,start_date equal to end_date,0.0
3,active (blank end_date),2355.0
4,active share %,67.2
5,ended,1147.0
6,ended before the window (end_date <= 2022-01-01),267.0
7,end_date not on the 1st of a month,0.0
8,customer x product with more than one active line,0.0
9,lines starting before the customer signup_date,0.0


,contract_term_months,lines,active_lines,share_pct
0,12,2086,1410,59.6
1,24,924,606,26.4
2,36,492,339,14.0


,product_id,product_name,list_price,lines,min_price,median_price,max_price,min_discount,median_discount,max_discount
0,1,Core Platform - Starter,480.0,677,444.44,475.93,480.00,0.0,0.0085,0.0741
1,2,Core Platform - Growth,1150.0,1069,916.97,1112.49,1230.50,0.0,0.0330,0.2026
2,3,Core Platform - Scale,2900.0,446,2347.85,2661.45,134166.67,0.0,0.0846,0.2633
3,4,Analytics Add-on,550.0,802,445.28,529.53,42500.00,0.0,0.0431,0.2633
4,5,Payments Module,720.0,508,574.10,681.46,42500.00,0.0,0.0584,0.2633


In [13]:
# 10. Management accounts
mc = q(con, "SELECT * FROM prof_mgmt_consistency")
bad = mc[(mc.lines_less_total.abs() > 0.005) | (mc.gp_arithmetic_diff.abs() > 0.005) | (mc.margin_pct_diff.abs() > 0.00005)]
doc.append("## 10. Management accounts: internal consistency\n")
doc.append("Months where the four product-line revenue columns do not add up to `total_revenue`, "
           "or gross profit or margin arithmetic fails:\n")
doc.append(md_table(bad, {"sum_of_lines": money2, "total_revenue": money2, "lines_less_total": money2,
                          "gp_arithmetic_diff": money2}))
doc.append(f"""
- {len(bad)} of {len(mc)} months fail the line-sum check. The differences are round amounts; they net to {gbp(bad.lines_less_total.sum())} across the four months and {gbp(bad.lines_less_total.abs().sum())} in absolute terms.
- `gross_profit` equals `total_revenue - cost_of_sales` in all {len(mc)} months, and `gross_margin_pct` equals `gross_profit / total_revenue` to four decimal places in all {len(mc)} months.
- These checks do not say which side (the lines or the total) is wrong. Step 3 compares both against the invoice data.
""")
bad

,month,sum_of_lines,total_revenue,lines_less_total,gp_arithmetic_diff,margin_pct_diff
17,2023-06-01,2735429.59,2716929.59,18500.0,0.0,0.0
26,2024-03-01,3039622.66,3051622.66,-12000.0,0.0,0.0
34,2024-11-01,3200376.09,3190876.09,9500.0,0.0,0.0
37,2025-02-01,3787073.54,3802073.54,-15000.0,0.0,0.0


In [14]:
# 11. Costs
cc = q(con, "SELECT * FROM prof_costs_completeness")
cv = q(con, "SELECT * FROM prof_costs_vs_cos")
cvy = cv.assign(year=cv.month.dt.year).groupby("year")[["cost_of_delivery", "cost_of_sales", "difference"]].sum().reset_index()
doc.append("## 11. Costs\n")
doc.append(md_table(cc, {"component_less_total": money2}))
doc.append("\nMonthly total cost of delivery against management `cost_of_sales`, summed by year:\n")
doc.append(md_table(cvy, {"cost_of_delivery": gbp, "cost_of_sales": gbp, "difference": money2}))
doc.append(f"""
- Every month x product line is present ({int(cc.present_cells.iloc[0])} of {int(cc.expected_cells.iloc[0])}).
- The three cost components add up to `total_cost_of_delivery` on every row to within a penny ({gbp(cc.component_less_total.iloc[0], 2)} across all rows, from rounding).
- Summed across product lines, costs equal management `cost_of_sales` in {int((cv.difference.abs() < 0.005).sum())} of {len(cv)} months.
""")
display(cc); display(cvy)

,expected_cells,present_cells,missing_cells,cost_rows,component_mismatches_over_1p,component_less_total
0,192,192,0,192,0,0.04


,year,cost_of_delivery,cost_of_sales,difference
0,2022,6367072.32,6367072.32,0.0
1,2023,7276948.58,7276948.58,0.0
2,2024,8276093.15,8276093.15,0.0
3,2025,12008133.71,12008133.71,0.0


In [15]:
# 12. Customer names and missing attributes
nf = q(con, "SELECT * FROM prof_name_formats")
ni = q(con, "SELECT * FROM prof_name_irregular")
sp_ = q(con, "SELECT * FROM prof_name_shared_pairs")
ss = q(con, "SELECT * FROM prof_name_shared_summary")
mi = q(con, "SELECT * FROM prof_missing_industry").iloc[0]
doc.append("## 12. Customer names and missing attributes\n")
doc.append("""Matching key (`customer_name_key` macro in `sql/01_profile.sql`, reused in cleaning): trim and collapse spaces, strip trailing punctuation, upper-case, then Limited to Ltd, Grp to Group, "and" to "&", and drop full stops after Co, PLC, Ltd and LLP.\n""")
doc.append(md_table(nf))
doc.append("\nNames that the key changes beyond case and legal form:\n")
doc.append(md_table(ni))
doc.append(f"""
- {len(ni)} names carry trailing punctuation or "and" in place of "&". None of them shares its key with another customer_id.
- {len(sp_)} pairs of different customer_ids ({sp_.id_a.nunique() + sp_.id_b.nunique()} ids) share a key. In every pair the only difference in the raw name is Ltd against Limited: {int((sp_.name_a.str.replace('Limited', 'Ltd') == sp_.name_b.str.replace('Limited', 'Ltd')).sum())} of {len(sp_)} pairs become identical when Limited is replaced by Ltd.
- None of the pairs has the same signup date. Signups within a pair are a median of {int((sp_.signup_a - sp_.signup_b).abs().dt.days.median()):,} days apart.
- {int(sp_.subscriptions_overlap.sum())} of the {len(sp_)} pairs have subscription lines running at the same time.
- How often the two ids in a pair agree on each attribute, against two customers picked at random:
""")
doc.append(md_table(ss, {"shared_name_pairs_agree_pct": pct, "random_pairs_agree_pct": pct}))
doc.append(f"""
- `industry` is missing for {int(mi.customers)} customers, {int(mi.with_invoices)} of whom have invoices in the window ({gbp(mi.invoiced_amount)}, {pct(mi.share_of_invoiced_pct)} of invoiced value).
""")
doc.append("<details><summary>All shared-name pairs</summary>\n")
doc.append(md_table(sp_))
doc.append("\n</details>\n")
display(ni); display(ss)

,customer_id,customer_name,name_key,other_ids_with_same_key
0,C1274,Quorn Systems and Co,QUORN SYSTEMS & CO,0
1,C1402,Pemberton Foods PLC.,PEMBERTON FOODS PLC,0
2,C1707,Thornbury Holdings PLC.,THORNBURY HOLDINGS PLC,0
3,C2719,Merton Labs PLC.,MERTON LABS PLC,0
4,C2762,Ashcombe Group LLP.,ASHCOMBE GROUP LLP,0


,attribute,pairs,shared_name_pairs_agree_pct,random_pairs_agree_pct
0,account_manager,229,9.6,12.6
1,acquisition_channel,229,19.2,21.6
2,company_size,229,34.5,30.9
3,industry,229,9.6,9.2
4,region,229,9.2,11.5


In [16]:
# Issue register (top of the document)
iss = q(con, "SELECT * FROM prof_issues")
money0 = {c: gbp for c in ["amount_net", "amount_gross_abs", "net_2022", "net_2023", "net_2024", "net_2025"]}
top = ["# Data profile", "",
       "Generated by `notebooks/02_data_preparation.ipynb` from the `prof_*` tables in `sql/01_profile.sql`. "
       "Do not edit by hand: rerun `python run_all.py`.", "",
       "Source: the six data-room CSVs in `data/raw/`, loaded by `sql/00_load.sql`. Data window: January 2022 to December 2025. "
       "All amounts in £.", "",
       "## Issue register", "",
       "Items that touch **amounts** could change revenue totals if adjusted; they are listed first, largest first, "
       "by the absolute £ of the rows affected. Items that touch **attributes** affect classification, timing or cash "
       "but not revenue totals. Amounts by year are net. The register orders the findings for review and makes "
       "no judgement on treatment.", "",
       md_table(iss, money0), "",
       "Detail for each item is in the numbered sections below: P01 and P02 in section 8, P03 and P06 in section 7, "
       "P04 in section 6, P05 in section 10, P07 and P11 in section 12, P08 in section 1, P09 in section 3, P10 in section 2.", "",
       "## Checks with no exceptions", "",
       "- Primary keys are unique and never null in all six tables.",
       "- No orphan customer or product ids in invoices or subscriptions.",
       "- Invoice `revenue_type` matches the product's `revenue_type` on every row.",
       "- No invoice, cost or management-accounts date outside the window.",
       "- No subscription starts after it ends; every invoiced recurring month is covered by a subscription line.",
       "- Costs are complete (every month x product line) and tie to management `cost_of_sales` every month.",
       "- Management gross profit and gross margin arithmetic holds every month.",
       ""]
text = "\n".join(top) + "\n" + "\n".join(doc) + "\n"
(DOCS / "data_profile.md").write_text(text, encoding="utf-8")
print(f"wrote docs/data_profile.md ({len(text.splitlines())} lines)")
iss

wrote docs/data_profile.md (771 lines)


,issue_id,touches,table_name,issue,rows_affected,amount_net,amount_gross_abs,net_2022,net_2023,net_2024,net_2025
0,P01,amount,invoices,Recurring invoices above the product-level outlier fence (Rule A),780,38065512.38,38065512.38,9365000.04,9429166.71,9722999.88,9548345.75
1,P02,amount,invoices,Implementation Services invoices billed outside the customer's signup month,45,4076574.31,4076574.31,0.00,0.00,0.00,4076574.31
2,P03,amount,invoices,Credit notes (status credited),1084,-1336469.07,1336469.07,-329204.58,-277053.69,-321374.32,-408836.48
3,P04,amount,invoices,Exact duplicate invoices (excess copies only),40,96957.26,96957.26,8596.92,70158.24,7115.70,11086.40
4,P05,amount,management_accounts,Product-line revenue does not sum to total_revenue,4,1000.00,55000.00,0.00,18500.00,-2500.00,-15000.00
5,P06,amount,invoices,Negative amounts on invoices marked paid or overdue,15,-13843.82,13843.82,-5788.75,-2118.96,-4073.48,-1862.63
6,P07,attribute,customers,Customer IDs that share a name once legal forms are normalised (their invoices),19105,22523721.17,22969347.65,3925704.04,4872061.29,5655589.75,8070366.09
7,P08,attribute,customers,Customers with missing industry (their invoices),6039,6594149.82,6739977.30,1233170.03,1423436.74,1829037.24,2108505.81
8,P09,attribute,invoices,Invoices with status overdue,2788,4463774.53,4463774.53,918707.94,915646.20,1092445.57,1536974.82
9,P10,attribute,invoices,Invoices dated before the customer's signup_date,1424,2992625.42,3016007.26,603914.63,772503.47,780330.04,835877.28


## 2.3 Clean

`sql/02_clean.sql` builds the `clean_*` tables. The only rows removed are the 40 exact duplicate copies. Everything else is carried with flags so the analysis can separate it. One row per affected record goes into `cleaning_actions`, and the cleaning log below is generated from that table.

In [17]:
rules = q(con, "SELECT * FROM cleaning_rules ORDER BY action_id")
acts = q(con, "SELECT * FROM cleaning_actions")
acts["year"] = pd.to_datetime(acts.record_date).dt.year
years = [2022, 2023, 2024, 2025]

summary = (acts.groupby(["action_id", "issue_id", "table_name"]).size().rename("records").reset_index())
for y in years:
    summary[f"impact_{y}"] = summary.action_id.map(acts[acts.year == y].groupby("action_id").revenue_impact.sum()).fillna(0)
summary["impact_total"] = summary[[f"impact_{y}" for y in years]].sum(axis=1)
summary = summary.merge(rules[["action_id", "action_type"]], on="action_id")
summary = summary[["action_id", "issue_id", "table_name", "action_type", "records"] + [f"impact_{y}" for y in years] + ["impact_total"]]
summary

,action_id,issue_id,table_name,action_type,records,impact_2022,impact_2023,impact_2024,impact_2025,impact_total
0,A01,P04,invoices,removed,40,-8596.92,-70158.24,-7115.7,-11086.4,-96957.26
1,A02,P06,invoices,flagged,15,0.00,0.00,0.0,0.0,0.00
2,A03,P03,invoices,linked,1084,0.00,0.00,0.0,0.0,0.00
3,A04,P08,customers,filled,132,0.00,0.00,0.0,0.0,0.00
4,A05,P07,customers,flagged,458,0.00,0.00,0.0,0.0,0.00
5,A06,P11,customers,standardised,382,0.00,0.00,0.0,0.0,0.00
6,A07,P02,invoices,categorised,45,0.00,0.00,0.0,0.0,0.00
7,A08,P05,management_accounts,flagged,4,0.00,0.00,0.0,0.0,0.00
8,A09,P10,invoices,flagged,1423,0.00,0.00,0.0,0.0,0.00
9,A10,P01,customers,flagged,6,0.00,0.00,0.0,0.0,0.00


In [18]:
# Row reconciliation: raw = clean + removed, per table
rowrec = q(con, """
    SELECT 'customers' AS table_name, (SELECT count(*) FROM raw_customers) AS raw_rows, (SELECT count(*) FROM clean_customers) AS clean_rows, 0 AS removed_rows
    UNION ALL SELECT 'products', (SELECT count(*) FROM raw_products), (SELECT count(*) FROM clean_products), 0
    UNION ALL SELECT 'subscriptions', (SELECT count(*) FROM raw_subscriptions), (SELECT count(*) FROM clean_subscriptions), 0
    UNION ALL SELECT 'invoices', (SELECT count(*) FROM raw_invoices), (SELECT count(*) FROM clean_invoices), (SELECT count(*) FROM clean_removed_invoices)
    UNION ALL SELECT 'costs', (SELECT count(*) FROM raw_costs), (SELECT count(*) FROM clean_costs), 0
    UNION ALL SELECT 'management_accounts', (SELECT count(*) FROM raw_management_accounts), (SELECT count(*) FROM clean_management_accounts), 0
""")
rowrec["check"] = rowrec.raw_rows == rowrec.clean_rows + rowrec.removed_rows
rowrec

,table_name,raw_rows,clean_rows,removed_rows,check
0,customers,2192,2192,0,True
1,products,6,6,0,True
2,subscriptions,3502,3502,0,True
3,invoices,92732,92692,40,True
4,costs,192,192,0,True
5,management_accounts,48,48,0,True


In [19]:
# P04: removed copies against the kept originals
dupcmp = q(con, """
    SELECT year(r.invoice_date) AS year, count(*) AS copies_removed, sum(r.amount) AS amount_removed,
           count(*) FILTER (WHERE r.status = k.status) AS same_status_as_original,
           count(*) FILTER (WHERE k.status = 'paid') AS original_paid,
           count(*) FILTER (WHERE k.status = 'overdue') AS original_overdue
    FROM clean_removed_invoices r JOIN clean_invoices k ON k.invoice_id = r.kept_invoice_id
    GROUP BY 1 ORDER BY 1
""")
dupcmp

,year,copies_removed,amount_removed,same_status_as_original,original_paid,original_overdue
0,2022,9,8596.92,9,0,9
1,2023,9,70158.24,9,0,9
2,2024,8,7115.70,8,0,8
3,2025,14,11086.40,14,0,14


In [20]:
# P09: overdue by year, for the working-capital team
overdue = q(con, """
    SELECT year(month_start) AS year,
           sum(amount) FILTER (WHERE NOT is_credit_note) AS gross_billings,
           count(*) FILTER (WHERE is_overdue) AS overdue_invoices,
           sum(amount) FILTER (WHERE is_overdue) AS overdue_amount,
           round(100.0 * sum(amount) FILTER (WHERE is_overdue) / sum(amount) FILTER (WHERE NOT is_credit_note), 2) AS overdue_pct,
           coalesce(sum(amount) FILTER (WHERE is_overdue AND oneoff_category = 'other_implementation'), 0) AS of_which_other_implementation
    FROM clean_invoices GROUP BY 1 ORDER BY 1
""")
overdue.to_csv(TABLES / "step2_overdue_by_year.csv", index=False)
overdue

,year,gross_billings,overdue_invoices,overdue_amount,overdue_pct,of_which_other_implementation
0,2022,28555567.47,513,910111.02,3.19,0.00
1,2023,32467804.68,661,845487.96,2.60,0.00
2,2024,37576199.35,757,1085329.87,2.89,0.00
3,2025,46163393.20,817,1525888.42,3.31,277704.19


In [21]:
# docs/cleaning_log.md
money = {c: gbp for c in [f"impact_{y}" for y in years] + ["impact_total"]}
log = ["# Cleaning log", "",
       "Generated by `notebooks/02_data_preparation.ipynb` from the `cleaning_rules` and `cleaning_actions` tables in "
       "`sql/02_clean.sql`. Do not edit by hand: rerun `python run_all.py`.", "",
       "Principle: cleaning fixes data errors only. Genuine revenue stays in the data with flags, and revenue-quality "
       "judgements are made in the analysis. Issue ids (P01 to P11) refer to `docs/data_profile.md`; question ids (Q01 onwards) "
       "refer to `docs/qa_log.md`.", "",
       "## Summary", "",
       "£ impact is the change to net revenue in the cube under the current anomaly setting "
       f"(`{q(con, 'SELECT anomaly_treatment FROM cfg_settings').iloc[0, 0]}`), by the year of the record.", "",
       md_table(summary, money), "",
       "Row reconciliation (raw rows = clean rows + removed rows):", "",
       md_table(rowrec), "",
       "## Adjustments", ""]
for _, r in rules.iterrows():
    a = acts[acts.action_id == r.action_id]
    by_year = ", ".join(f"{y}: {gbp(a[a.year == y].revenue_impact.sum(), 2)}" for y in years)
    log += [f"### {r.action_id} ({r.issue_id}): {r.action_type}", "",
            f"- **Table:** `{r.table_name}`",
            f"- **Rule:** {r.rule}",
            f"- **Records affected:** {len(a):,}",
            f"- **£ impact on net revenue by year:** {by_year}",
            f"- **Rationale:** {r.rationale}",
            f"- **Alternative considered:** {r.alternative_considered}", ""]
    if r.action_id == "A01":
        log += ["Removed copies against the originals kept:", "", md_table(dupcmp, {"amount_removed": lambda v: gbp(v, 2)}), "",
                "Originals and copies carry the same status in every case, so no copy sits behind a paid original. "
                "Both copies of each pair are marked overdue, so the overdue balance in the source includes the removed "
                "copies (Q06).", ""]
    if r.action_id == "A02":
        log += [f"- **Alternative impacts:** flipping the sign would add {gbp(a.impact_if_flipped.sum(), 2)} to net revenue; "
                f"excluding the rows would add {gbp(a.impact_if_excluded.sum(), 2)}. Step 3 decides.", ""]
log += ["## Reported, not adjusted", "",
        "P09 overdue invoices stay in revenue. Overdue £ and % of gross billings by year "
        "(also saved to `outputs/tables/step2_overdue_by_year.csv`):", "",
        md_table(overdue, {"gross_billings": gbp, "overdue_amount": gbp, "overdue_pct": pct,
                           "of_which_other_implementation": gbp}), ""]
(DOCS / "cleaning_log.md").write_text("\n".join(log), encoding="utf-8")
print("wrote docs/cleaning_log.md")

wrote docs/cleaning_log.md


## 2.4 Revenue cube

`sql/03_revenue_cube.sql` builds `fact_revenue_monthly` (customer x product x month), `fact_mrr_monthly` (recurring billings before credit notes) and three dimensions, and exports them to `data/clean/`. The P06 treatment is one setting, `cfg_settings.anomaly_treatment`. The macros `revenue_cube(t)` and `mrr_cube(t)` rebuild the facts for any treatment, which is what Step 3 uses to choose.

In [22]:
cube_rows = q(con, """
    SELECT 'fact_revenue_monthly' AS table_name, count(*) AS row_count FROM fact_revenue_monthly
    UNION ALL SELECT 'fact_mrr_monthly', count(*) FROM fact_mrr_monthly
    UNION ALL SELECT 'dim_customer', count(*) FROM dim_customer
    UNION ALL SELECT 'dim_product', count(*) FROM dim_product
    UNION ALL SELECT 'dim_date', count(*) FROM dim_date
""")
cube_rows

,table_name,row_count
0,fact_revenue_monthly,91608
1,fact_mrr_monthly,90388
2,dim_customer,2192
3,dim_product,6
4,dim_date,48


In [23]:
# Source-to-cube walk by year: raw gross billings -> less duplicates -> anomaly treatment -> credit notes -> cube net
walk = q(con, """
    WITH raw AS (SELECT year(invoice_date) AS year,
                        sum(amount) FILTER (WHERE status <> 'credited') AS raw_gross_billings,
                        sum(amount) FILTER (WHERE status = 'credited') AS credit_notes
                 FROM raw_invoices GROUP BY 1),
         dup AS (SELECT year(invoice_date) AS year, sum(amount) AS duplicates FROM clean_removed_invoices GROUP BY 1),
         anom AS (SELECT year(month_start) AS year,
                         sum(invoice_amount((SELECT anomaly_treatment FROM cfg_settings), amount_as_reported, amount_flipped, amount_excluded) - amount)
                           FILTER (WHERE anomaly_flag) AS anomaly_adjustment
                  FROM clean_invoices GROUP BY 1),
         cube AS (SELECT year(month_start) AS year, sum(net_revenue) AS cube_net FROM fact_revenue_monthly GROUP BY 1)
    SELECT raw.year, raw.raw_gross_billings, -coalesce(dup.duplicates, 0) AS less_duplicates,
           coalesce(anom.anomaly_adjustment, 0) AS anomaly_treatment, raw.credit_notes, cube.cube_net
    FROM raw LEFT JOIN dup USING (year) LEFT JOIN anom USING (year) JOIN cube USING (year)
    ORDER BY raw.year
""")
walk["walk_check"] = (walk.raw_gross_billings + walk.less_duplicates + walk.anomaly_treatment + walk.credit_notes - walk.cube_net).abs() < 0.005
walk.to_csv(TABLES / "step2_source_to_cube_walk.csv", index=False)
walk

,year,raw_gross_billings,less_duplicates,anomaly_treatment,credit_notes,cube_net,walk_check
0,2022,28564164.39,-8596.92,0.0,-329204.58,28226362.89,True
1,2023,32537962.92,-70158.24,0.0,-277053.69,32190750.99,True
2,2024,37583315.05,-7115.70,0.0,-321374.32,37254825.03,True
3,2025,46174479.60,-11086.40,0.0,-408836.48,45754556.72,True


In [24]:
# Cube net under each anomaly treatment, and cube against management reported revenue (numbers only; Step 3 reconciles)
opts = q(con, """
    SELECT year(month_start) AS year, t AS treatment, sum(net_revenue) AS cube_net
    FROM (SELECT 'as_reported' AS t, * FROM revenue_cube('as_reported')
          UNION ALL SELECT 'flipped', * FROM revenue_cube('flipped')
          UNION ALL SELECT 'excluded', * FROM revenue_cube('excluded'))
    GROUP BY ALL
""").pivot(index="year", columns="treatment", values="cube_net").reset_index()
mgmt = q(con, """SELECT year(month) AS year, sum(total_revenue) AS mgmt_total_revenue,
                        sum(revenue_sum_of_lines) AS mgmt_sum_of_lines FROM clean_management_accounts GROUP BY 1""")
vs = opts.merge(mgmt, on="year")[["year", "as_reported", "flipped", "excluded", "mgmt_total_revenue", "mgmt_sum_of_lines"]]
vs["as_reported_less_mgmt_total"] = vs.as_reported - vs.mgmt_total_revenue
vs.to_csv(TABLES / "step2_cube_vs_management.csv", index=False)
vs

,year,as_reported,flipped,excluded,mgmt_total_revenue,mgmt_sum_of_lines,as_reported_less_mgmt_total
0,2022,28226362.89,28237940.39,28232151.64,28232151.64,28232151.64,-5788.75
1,2023,32190750.99,32194988.91,32192869.95,32174369.95,32192869.95,16381.04
2,2024,37254825.03,37262971.99,37258898.51,37261398.51,37258898.51,-6573.48
3,2025,45754556.72,45758281.98,45756419.35,45771419.35,45756419.35,-16862.63


In [25]:
mrr_cn = q(con, "SELECT * FROM mrr_credit_notes_excluded")
mrr_year = q(con, "SELECT year(month_start) AS year, sum(mrr) AS mrr_total, count(DISTINCT customer_id) AS customers_with_mrr FROM fact_mrr_monthly GROUP BY 1 ORDER BY 1")
display(mrr_cn); display(mrr_year)

,year,credit_notes,credit_note_amount
0,2022,201,-329204.58
1,2023,258,-277053.69
2,2024,288,-321374.32
3,2025,337,-408836.48


,year,mrr_total,customers_with_mrr
0,2022,27571707.88,1093
1,2023,31289600.88,1316
2,2024,36431984.53,1498
3,2025,40882071.00,1650


## 2.5 Documentation

The data dictionary is built from the column comments in the SQL files. Where a derived table repeats a column from a clean table without its own comment, the clean table's description is used. The Q&A log is built from the `qa_log` table.

In [26]:
# docs/data_dictionary.md
cols = q(con, """
    SELECT table_name, column_index, column_name, data_type, comment FROM duckdb_columns()
    WHERE starts_with(table_name, 'clean_') OR starts_with(table_name, 'fact_') OR starts_with(table_name, 'dim_') OR table_name IN ('cleaning_actions', 'qa_log')
    ORDER BY table_name, column_index
""")
fallback = {}
for src in ["clean_customers", "clean_invoices", "clean_products"]:
    for _, r in cols[(cols.table_name == src) & cols.comment.notna()].iterrows():
        fallback.setdefault(r.column_name, r.comment)
cols["description"] = cols.comment.where(cols.comment.notna() & (cols.comment != ""), cols.column_name.map(fallback))
qa_desc = {"qa_id": "Question id (Q01 onwards).", "step": "Step that raised the question.", "issue_ref": "Profile issue or analysis reference.",
           "topic": "Topic.", "question": "Question for management.", "evidence": "What the data shows, with numbers built from the tables.",
           "status": "open or answered."}
cols.loc[cols.table_name == "qa_log", "description"] = cols.column_name.map(qa_desc)
missing = cols[cols.description.isna()]
assert missing.empty, f"columns without a description: {missing[['table_name', 'column_name']].values.tolist()}"

table_notes = {
    "clean_customers": "One row per customer. Source: raw_customers plus subscription-derived fields.",
    "clean_products": "One row per product. Unchanged from raw_products.",
    "clean_subscriptions": "One row per contract line. raw_subscriptions plus flags.",
    "clean_invoices": "One row per billing event after removing the 40 duplicate copies (A01).",
    "clean_removed_invoices": "The rows removed from invoices, with the id of the original kept.",
    "clean_costs": "One row per month x product line. raw_costs plus a component check.",
    "clean_management_accounts": "One row per month. raw_management_accounts plus the line-sum check (A08).",
    "fact_revenue_monthly": "One row per customer x product x month with any billing or credit. Grain key: customer_id, product_id, month_start.",
    "fact_mrr_monthly": "One row per customer x recurring product x month with non-zero MRR. Recurring billings before credit notes.",
    "dim_customer": "One row per customer. Customer attributes, cohort fields and flags.",
    "dim_product": "One row per product.",
    "dim_date": "One row per month, January 2022 to December 2025.",
    "cleaning_actions": "One row per record affected by a cleaning rule. Feeds docs/cleaning_log.md.",
    "qa_log": "Questions for management. Feeds docs/qa_log.md.",
}
dd = ["# Data dictionary", "",
      "Generated by `notebooks/02_data_preparation.ipynb` from the column comments in `sql/02_clean.sql` and "
      "`sql/03_revenue_cube.sql`. Do not edit by hand. Raw tables mirror the CSVs in `data/raw/`; see `docs/data_profile.md`.", ""]
for t in table_notes:
    g = cols[cols.table_name == t]
    dd += [f"## {t}", "", table_notes[t], "", md_table(g[["column_name", "data_type", "description"]]), ""]
(DOCS / "data_dictionary.md").write_text("\n".join(dd), encoding="utf-8")
print(f"wrote docs/data_dictionary.md ({len(cols)} columns)")

wrote docs/data_dictionary.md (162 columns)


In [27]:
# docs/qa_log.md
qa = q(con, "SELECT * FROM qa_log ORDER BY qa_id")
ql = ["# Q&A log", "",
      "Questions for management raised by the data. Generated from the `qa_log` table, which each step's SQL appends to. "
      "The evidence is built from the tables on every run. Do not edit by hand.", ""]
for _, r in qa.iterrows():
    ql += [f"## {r.qa_id}: {r.topic} ({r.issue_ref}, {r.step})", "",
           f"**Question.** {r.question}", "",
           f"**Evidence.** {r.evidence}", "",
           f"**Status.** {r.status}", ""]
(DOCS / "qa_log.md").write_text("\n".join(ql), encoding="utf-8")
print(f"wrote docs/qa_log.md ({len(qa)} questions)")
qa[["qa_id", "issue_ref", "topic"]]

wrote docs/qa_log.md (10 questions)


,qa_id,issue_ref,topic
0,Q01,P07,Customer identity
1,Q02,P02,Other implementation revenue
2,Q03,P02,Other implementation revenue
3,Q04,P02,Other implementation revenue
4,Q05,P02,Other implementation revenue
5,Q06,P04,Duplicate invoices
6,Q07,P05,Management accounts
7,Q08,P06,Negative invoices
8,Q09,P10,Contract dates
9,Q10,P08,Customer attributes


In [28]:
# Generated blocks in docs/metric_definitions.md
replace_block(DOCS / "metric_definitions.md", "mrr_credit_notes",
              md_table(mrr_cn.assign(credit_note_amount=mrr_cn.credit_note_amount),
                       {"credit_note_amount": gbp}) +
              f"\n\nTotal left out of MRR across 2022 to 2025: {gbp(mrr_cn.credit_note_amount.sum())} on {int(mrr_cn.credit_notes.sum()):,} credit notes.")
la = q(con, """
    SELECT count(DISTINCT s.customer_id) FILTER (WHERE s.monthly_price >= 10 * p.list_price) AS large_customers,
           count(*) FILTER (WHERE s.monthly_price >= 10 * p.list_price) AS large_lines,
           round(max(s.monthly_price / p.list_price) FILTER (WHERE s.monthly_price < 10 * p.list_price), 2) AS max_multiple_other_lines,
           round(min(s.monthly_price / p.list_price) FILTER (WHERE s.monthly_price >= 10 * p.list_price), 1) AS min_multiple_large_lines
    FROM clean_subscriptions s JOIN clean_products p USING (product_id)
""").iloc[0]
replace_block(DOCS / "metric_definitions.md", "large_accounts",
              f"The rule flags {int(la.large_customers)} customers ({int(la.large_lines)} subscription lines, priced at "
              f"{la.min_multiple_large_lines:.1f} times list or more). Every other subscription line is priced at no more than "
              f"{la.max_multiple_other_lines:.2f} times list, so the 10x threshold separates the two groups with a wide margin.")
print("updated docs/metric_definitions.md")

updated docs/metric_definitions.md


In [29]:
# Generated check totals in docs/alteryx_workflow.md
ct_load = q(con, """
    SELECT 'raw_customers' AS stream, count(*) AS row_count, NULL::DOUBLE AS amount_total FROM raw_customers
    UNION ALL SELECT 'raw_products', count(*), NULL FROM raw_products
    UNION ALL SELECT 'raw_subscriptions', count(*), sum(monthly_price) FROM raw_subscriptions
    UNION ALL SELECT 'raw_invoices', count(*), sum(amount) FROM raw_invoices
    UNION ALL SELECT 'raw_costs', count(*), sum(total_cost_of_delivery) FROM raw_costs
    UNION ALL SELECT 'raw_management_accounts', count(*), sum(total_revenue) FROM raw_management_accounts
""")
ct_prof = q(con, """
    SELECT 'duplicate primary keys (all tables)' AS check_name, (SELECT sum(row_count - distinct_keys) FROM prof_primary_keys)::DOUBLE AS value
    UNION ALL SELECT 'orphan ids in invoices and subscriptions', (SELECT sum(orphan_rows) FROM prof_referential_integrity WHERE check_name LIKE '%not in%')
    UNION ALL SELECT 'exact duplicate groups', (SELECT count(*) FROM prof_duplicate_invoices)
    UNION ALL SELECT 'negative non-credit invoices', (SELECT count(*) FROM prof_negative_non_credit)
    UNION ALL SELECT 'credit notes unmatched', (SELECT sum(credit_notes - matched_to_original) FROM prof_credit_notes)
    UNION ALL SELECT 'recurring invoices above the Rule A fence', (SELECT sum(invoices_above) FROM prof_outlier_fences WHERE product_id <> '6')
    UNION ALL SELECT 'management months where lines do not add up', (SELECT count(*) FROM prof_mgmt_consistency WHERE abs(lines_less_total) > 0.005)
    UNION ALL SELECT 'months where costs do not equal cost_of_sales', (SELECT count(*) FROM prof_costs_vs_cos WHERE abs(difference) > 0.005)
""")
ct_clean = q(con, """
    SELECT 'clean_invoices rows' AS check_name, count(*)::DOUBLE AS value FROM clean_invoices
    UNION ALL SELECT 'clean_invoices amount total', sum(amount) FROM clean_invoices
    UNION ALL SELECT 'removed duplicate rows', (SELECT count(*) FROM clean_removed_invoices)
    UNION ALL SELECT 'removed duplicate amount', (SELECT sum(amount) FROM clean_removed_invoices)
    UNION ALL SELECT 'credit notes linked to an original', count(*) FILTER (WHERE credit_note_original_id IS NOT NULL) FROM clean_invoices
    UNION ALL SELECT 'anomaly_flag rows', count(*) FILTER (WHERE anomaly_flag) FROM clean_invoices
    UNION ALL SELECT 'other_implementation rows', count(*) FILTER (WHERE oneoff_category = 'other_implementation') FROM clean_invoices
    UNION ALL SELECT 'other_implementation amount', sum(amount) FILTER (WHERE oneoff_category = 'other_implementation') FROM clean_invoices
    UNION ALL SELECT 'before_signup_flag rows', count(*) FILTER (WHERE before_signup_flag) FROM clean_invoices
    UNION ALL SELECT 'customers with industry Unknown', (SELECT count(*) FROM clean_customers WHERE industry = 'Unknown')
    UNION ALL SELECT 'customers with name_collision_flag', (SELECT count(*) FROM clean_customers WHERE name_collision_flag)
    UNION ALL SELECT 'customers with is_large_account', (SELECT count(*) FROM clean_customers WHERE is_large_account)
    UNION ALL SELECT 'management months flagged', (SELECT count(*) FROM clean_management_accounts WHERE lines_total_mismatch_flag)
""")
ct_cube_year = q(con, """
    SELECT year(f.month_start) AS year, sum(f.gross_billings) AS gross_billings, sum(f.credit_notes) AS credit_notes,
           sum(f.net_revenue) AS net_revenue, sum(f.recurring_net) AS recurring_net, sum(f.oneoff_net) AS oneoff_net,
           (SELECT sum(mrr) FROM fact_mrr_monthly m WHERE year(m.month_start) = year(f.month_start)) AS mrr_total
    FROM fact_revenue_monthly f GROUP BY 1 ORDER BY 1
""")
fmt_val = lambda v: num(v) if float(v).is_integer() else num(v, 2)
block = ["### Container 1: Load", "", md_table(ct_load, {"amount_total": lambda v: gbp(v, 2)}), "",
         "### Container 2: Profile", "", md_table(ct_prof, {"value": fmt_val}), "",
         "### Container 3: Clean", "", md_table(ct_clean, {"value": fmt_val}), "",
         "### Container 4: Revenue cube (anomaly setting as_reported)", "",
         md_table(cube_rows), "", md_table(ct_cube_year, {c: lambda v: gbp(v, 2) for c in ct_cube_year.columns if c != "year"})]
replace_block(DOCS / "alteryx_workflow.md", "check_totals", "\n".join(block))
print("updated docs/alteryx_workflow.md")

updated docs/alteryx_workflow.md
